# Stop a Runaway Agent Before It Burns Your Budget

This notebook shows how to catch a looping agent before it keeps making the same tool call and model call over and over. It uses only `agentdbg` plus a tiny local-only stub, so it runs without API keys, network calls, or framework dependencies.

## Setup

If not done already, install AgentDbg in your notebook environment (run once):

In [ ]:
# %pip install --upgrade pip -q
# %pip install agentdbg -q

In [ ]:
import os

from agentdbg import (
    AgentDbgLoopAbort,
    record_llm_call,
    record_state,
    record_tool_call,
    trace,
)

## Build a tiny local-only agent

We will simulate a very small agent with two moving parts:

- A deterministic `search_docs` tool that always returns no results.
- A deterministic "planner" response that always says to retry the same query.

That is enough to create the repeated `(TOOL_CALL, LLM_CALL)` pattern that AgentDbg can detect as a loop.

In [ ]:
os.environ.setdefault("AGENTDBG_LOOP_WINDOW", "6")
os.environ.setdefault("AGENTDBG_LOOP_REPETITIONS", "3")

QUERY = "current pricing tiers"
MODEL_NAME = "demo-model-local"
LOOP_RESPONSE = "The search returned nothing. Try again with the same query."


def search_docs(query: str) -> dict:
    return {"hits": [], "total": 0}


def scripted_next_step() -> str:
    return LOOP_RESPONSE


def simulate_runaway_agent(iterations: int) -> None:
    record_state(
        state={"phase": "init", "goal": "look up current pricing"},
        meta={"tutorial": "guardrails"},
    )

    for i in range(iterations):
        result = search_docs(QUERY)
        record_tool_call(
            name="search_docs",
            args={"query": QUERY, "filter": "public"},
            result=result,
            meta={"tutorial": "guardrails", "iteration": i},
        )
        record_llm_call(
            model=MODEL_NAME,
            prompt="The search returned no results. What should I do?",
            response=scripted_next_step(),
            usage={"prompt_tokens": 16, "completion_tokens": 12, "total_tokens": 28},
            meta={"tutorial": "guardrails", "iteration": i},
        )

## Let it loop without a guardrail

First we run the agent without `stop_on_loop`. The loop is still capped at 5 iterations so the notebook finishes quickly, but it runs long enough for AgentDbg to emit a `LOOP_WARNING`.

In [ ]:
@trace(name="Runaway agent (no guardrail)")
def run_without_guardrail() -> None:
    simulate_runaway_agent(iterations=5)


run_without_guardrail()
print("Run finished (no stop_on_loop). Check the timeline for LOOP_WARNING.")

In [ ]:
!agentdbg list

## Understand the warning

AgentDbg's loop detector looks at the last `N` events in the active run. With:

- `AGENTDBG_LOOP_WINDOW=6`
- `AGENTDBG_LOOP_REPETITIONS=3`

it checks whether the last 6 events are the same 2-event pattern repeated 3 times. In this notebook that pattern is:

1. `TOOL_CALL` for `search_docs`
2. `LLM_CALL` for `demo-model-local`

When that repeats three times in a row, AgentDbg emits a `LOOP_WARNING`. The run can still continue if you only want to observe the pattern.

## Stop the loop with `stop_on_loop`

Now we run the same logic with the guardrail enabled. This time AgentDbg raises `AgentDbgLoopAbort` as soon as the repeated pattern reaches the configured threshold, so the agent stops before it keeps spending more budget.

In [ ]:
@trace(
    name="Runaway agent (with guardrail)",
    stop_on_loop=True,
    stop_on_loop_min_repetitions=3,
    max_llm_calls=20,
    max_tool_calls=20,
)
def run_with_guardrail() -> None:
    simulate_runaway_agent(iterations=15)


try:
    run_with_guardrail()
    print("Unexpected: run did not abort.")
except AgentDbgLoopAbort as e:
    print(f"AgentDbg stopped the agent: {e}")
    print("The full trace is saved; open it with: agentdbg view")

In [ ]:
!agentdbg list

## View the timeline

In a terminal, run:

```bash
agentdbg view
```

Open the newest two runs from this notebook and compare them:

- `Runaway agent (no guardrail)` should contain a `LOOP_WARNING` but still finish.
- `Runaway agent (with guardrail)` should contain the same warning plus an early abort.
- In both runs, the timeline shows the exact tool and model calls that formed the loop evidence.

## Why this content play works

The point is not that the model is bad. The point is that even a simple retry loop can silently waste tool calls and model calls when nothing in the state changes. AgentDbg makes that repetition visible, and `stop_on_loop` turns the warning into a hard stop when you want protection instead of just observability.